# Bootstrap pseudo-bulk for single-nucleus RNA-seq

- Load merged AnnData (raw counts).
- Map samples to condition (DMD vs healthy).
- Generate bootstrap pseudo-bulk counts (e.g. 50 iterations, 90% of cells per condition) for downstream DESeq2.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore", message=".*The 'nopython' keyword.*")
import pandas as pd
import scanpy as sc
import os
import matplotlib.pyplot as plt
import seaborn as sns
seed = 42
np.random.seed(seed)
sc.settings.verbosity = 3
from scipy import sparse
import scrublet as scr
sc.settings.set_figure_params(dpi=80, facecolor='white')

## Load merged AnnData

Read the merged raw-count AnnData object (e.g. after QC and cell-type assignment).

In [ ]:
adata_merge_mac = sc.read_h5ad("./singlen/humanDMD_rawcount.h5ad")

## Sample-to-condition mapping

Map sample IDs to DMD vs healthy for downstream bootstrap and DESeq2.

In [ ]:
sample_mapping = {'0': 'DMD', '1': 'DMD', '2': 'DMD', '3': 'healthy','4': 'healthy'}
adata_merge_mac.obs['condition'] = adata_merge_mac.obs['sample'].replace(sample_mapping)

## Bootstrap pseudo-bulk

Subsample a fraction of cells per condition (with replacement across iterations), sum counts per gene, and write a table for DESeq2. Optional: restrict to a cell type (e.g. macrophages) via `is_mac`.

In [ ]:
# --- Output path and parameters ---
out_tsv = "//humanDMD/singlen/bootstrap/pseudo_bulk_mac_bootstrap50_90pct.txt"
B = 50
frac = 0.90

# --- Restrict to cell type (optional) and get condition masks ---
adata = adata_merge_mac
if "is_mac" in adata.obs.columns:
    adata = adata[adata.obs["is_mac"] == True].copy()

cond = adata.obs["condition"].astype(str)
mask_dmd = cond.str.lower() == "dmd"
mask_hlt = cond.str.lower() == "healthy"

X = adata.X.tocsr() if sparse.issparse(adata.X) else sparse.csr_matrix(adata.X)

# --- Bootstrap per condition and write table ---
def bootstrap_pseudobulk(X_csr, mask, frac, B, prefix):
    idx_all = np.where(mask)[0]
    n = idx_all.size
    k = max(1, int(np.floor(n * frac)))
    cols = []
    for b in range(1, B+1):
        sel = np.random.choice(idx_all, size=k, replace=False)
        s = X_csr[sel, :].sum(axis=0)  
        s = np.asarray(s).ravel()
        cols.append(pd.Series(s, name=f"{prefix}{b}"))
    return pd.concat(cols, axis=1)

df_dmd = bootstrap_pseudobulk(X, mask_dmd, frac, B, "DMD")
df_hlt = bootstrap_pseudobulk(X, mask_hlt, frac, B, "Healthy")

df = pd.concat([df_dmd, df_hlt], axis=1)
df.insert(0, "Geneid", adata.var_names.astype(str).tolist())
df.to_csv(out_tsv, sep="\t", index=False)
print("Saved to:", out_tsv, "| shape:", df.shape)

Saved to: /mnt/md0/jiahui/humanDMD/singlen/bootstrap/pseudo_bulk_mac_bootstrap50_90pct.txt | shape: (32738, 101)
